In [0]:
source_path = '/Volumes/finguard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

[FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131048_780991_0.json', name='fraud_watchlist_20260715_131048_780991_0.json', size=365, modificationTime=1784121050000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131054_583086_1.json', name='fraud_watchlist_20260715_131054_583086_1.json', size=379, modificationTime=1784121055000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131100_152329_2.json', name='fraud_watchlist_20260715_131100_152329_2.json', size=380, modificationTime=1784121061000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131105_595528_3.json', name='fraud_watchlist_20260715_131105_595528_3.json', size=383, modificationTime=1784121067000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131111_231708_4.json', na

In [0]:
input_stream =( spark.readStream.format("cloudFiles")
               .option("cloudFiles.format","json")
               .option("cloudFiles.schemaLocation", "/Volumes/finguard/source/fraud_watchlist/schema/")
               .option("cloudFiles.inferColumnTypes", "true")
               .load(source_path))

In [0]:
from pyspark.sql.functions import *
transform_df = (input_stream.select(
                "*",
                col("_metadata.file_path").alias('file_path'),
                current_timestamp().alias('ingestion_timestamp')
))

In [0]:
streaming_query = (transform_df.writeStream.format("delta")
 .outputMode("append")
 .option("checkpointLocation","/Volumes/finguard/source/fraud_watchlist/checkpoint/")
 .trigger(availableNow=True)
 .toTable("finguard.bronze.fraud_watchlist_batch_test")
 )

print(f"Query Started with query_id: {streaming_query.id}")

Query Started with query_id: ac6f1845-9c22-4f45-8b59-62390dfe21b8


In [0]:
%sql
select * from finguard.bronze.fraud_watchlist_batch_test

action,city,country,effective_from,entity_id,reason_code,reason_description,reported_by,reported_source,risk_level,status,watch_type,watchlist_id,_rescued_data,file_path,ingestion_timestamp
ADD,Bengaluru,India,18-Jun-2026 09:07:00,4260650191792026,ACCOUNT_TAKEOVER,Auto-generated fraud watchlist entry,Customer Care,CRM,critical,ACTIVE,CARD,wl000005,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131111_231708_4.json,2026-07-15T13:23:04.477Z
ADD,Chennai,India,18-Jun-2026 09:04:00,4717327980595032,TEST_TRANSACTION,Auto-generated fraud watchlist entry,AML Team,External Feed,high,ACTIVE,CARD,wl000004,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131105_595528_3.json,2026-07-15T13:23:04.477Z
ADD,Delhi,India,18-Jun-2026 09:02:00,5329839542961842,TEST_TRANSACTION,Auto-generated fraud watchlist entry,AML Team,External Feed,low,ACTIVE,CARD,wl000003,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131100_152329_2.json,2026-07-15T13:23:04.477Z
ADD,Pune,India,18-Jun-2026 09:01:00,4413337312552355,BLACKLISTED_MERCHANT,Auto-generated fraud watchlist entry,SOC,External Feed,high,ACTIVE,CARD,wl000002,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131054_583086_1.json,2026-07-15T13:23:04.477Z
ADD,Hyderabad,India,18-Jun-2026 09:00:00,5008514036965665,PHISHING,Auto-generated fraud watchlist entry,SOC,Manual,high,ACTIVE,CARD,wl000001,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260715_131048_780991_0.json,2026-07-15T13:23:04.477Z
